# CMT Production Training
**Bayesian-optimized parameters** — ready to run on Colab T4/A100.

Run cells top to bottom. Checkpoints are saved directly to Google Drive every epoch.

In [ ]:
# 1. Setup
!git clone https://github.com/C0d3Crush/xray-vessel-inpainting.git
%cd xray-vessel-inpainting
!pip install -q torch torchvision matplotlib pillow opencv-python scikit-image pycocotools pandas numpy scipy
print('Setup complete')

In [ ]:
# 2. Mount Google Drive and extract ARCADE dataset
from google.colab import drive
import os
drive.mount('/content/drive')

ARCADE_ZIP = '/content/drive/MyDrive/arcade.zip'
!unzip -q -o "$ARCADE_ZIP" -d /content/xray-vessel-inpainting/data/
!ln -sf /content/xray-vessel-inpainting/data/arcade/stenosis /content/xray-vessel-inpainting/data/arcade/syntax
!find data/arcade -name '*.json' | head -5
print('ARCADE dataset ready')

In [ ]:
# 3. Auto-detect ARCADE dataset paths (syntax preferred over stenosis)
import os, glob

def find_arcade_paths():
    candidates = []
    for root in ['/content/xray-vessel-inpainting', '/content', '.']:
        for match in glob.glob(f'{root}/**/train.json', recursive=True):
            if any(k in match for k in ('arcade', 'syntax', 'stenosis')):
                ann_dir = os.path.dirname(match)
                img_dir = os.path.join(os.path.dirname(ann_dir), 'images')
                if os.path.isdir(img_dir):
                    candidates.append((match, img_dir))
    candidates.sort(key=lambda c: (0 if 'syntax' in c[0] else 1))
    return candidates

train_candidates = find_arcade_paths()
if not train_candidates:
    raise RuntimeError('ARCADE dataset not found — make sure Cell 2 ran successfully.')

train_ann_path, train_img_dir = train_candidates[0]
TRAIN_ANN = train_ann_path
TRAIN_IMG = train_img_dir
VAL_ANN   = train_ann_path.replace('train/annotations/train.json', 'val/annotations/val.json')
VAL_IMG   = train_img_dir.replace('train/images', 'val/images')

print(f'TRAIN_IMG: {TRAIN_IMG}')
print(f'TRAIN_ANN: {TRAIN_ANN}')
print(f'VAL_IMG:   {VAL_IMG}')
print(f'VAL_ANN:   {VAL_ANN}')
if 'stenosis' in TRAIN_ANN and 'syntax' not in TRAIN_ANN:
    print('WARNING: stenosis annotations found — no vessel polygons!')

In [ ]:
# 4. Imports, device, and annotation sanity-check
import sys, os, csv, json, subprocess
from collections import defaultdict

import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name} ({props.total_memory / 1e9:.1f} GB)')

def check_annotations(ann_path, label='dataset'):
    pkl_path = ann_path.replace('.json', '.pkl')
    if os.path.exists(pkl_path):
        import pickle
        with open(pkl_path, 'rb') as f:
            cached = pickle.load(f)
        n = len(cached.get('image_ids', []))
        print(f'  {label}: {n} annotated images (from .pkl cache)')
        if n == 0:
            print(f'  WARNING: stale cache has 0 images — delete it: !rm {pkl_path}')
        return n
    with open(ann_path) as f:
        coco = json.load(f)
    anns_by_img = defaultdict(list)
    for a in coco.get('annotations', []):
        anns_by_img[a['image_id']].append(a)
    n = sum(1 for img in coco.get('images', []) if anns_by_img[img['id']])
    print(f'  {label}: {n} annotated / {len(coco.get("images", []))} total images')
    return n

print('Annotation check:')
train_n = check_annotations(TRAIN_ANN, 'train')
val_n   = check_annotations(VAL_ANN,   'val')
if train_n == 0 or val_n == 0:
    raise RuntimeError('Dataset has 0 annotated images — check the paths or delete stale .pkl files.')

In [ ]:
# 5. Production training with Bayesian-optimized parameters
import shutil, subprocess, sys, csv, os
from datetime import datetime

# --- Optimized hyperparameters ---
BEST_PARAMS = {
    'lr':                1e-4,
    'batch_size':        6,
    'patches_per_image': 11,
    'foreground_prob':   0.7737056704131198,
    'max_shapes':        6,
}

EPOCHS       = 25
INPUT_SIZE   = 64
LOCAL_OUTPUT = 'checkpoints'

# Each run gets its own timestamped folder on Drive — previous runs are never overwritten
RUN_ID       = datetime.now().strftime('%Y%m%d_%H%M')
DRIVE_OUTPUT = f'/content/drive/MyDrive/CMT/runs/{RUN_ID}'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

print(f'Run ID:  {RUN_ID}')
print(f'Drive:   {DRIVE_OUTPUT}')
print(f'epochs={EPOCHS}, lr={BEST_PARAMS["lr"]:.0e}, bs={BEST_PARAMS["batch_size"]}, '
      f'ppi={BEST_PARAMS["patches_per_image"]}, fg={BEST_PARAMS["foreground_prob"]:.4f}, '
      f'shapes={BEST_PARAMS["max_shapes"]}')
print()

cmd = [
    sys.executable, 'src/train.py',
    '--train_img',         TRAIN_IMG,
    '--train_ann',         TRAIN_ANN,
    '--val_img',           VAL_IMG,
    '--val_ann',           VAL_ANN,
    '--epochs',            str(EPOCHS),
    '--batch_size',        str(BEST_PARAMS['batch_size']),
    '--lr',                str(BEST_PARAMS['lr']),
    '--input_size',        str(INPUT_SIZE),
    '--device',            device,
    '--output_dir',        LOCAL_OUTPUT,
    '--patches_per_image', str(BEST_PARAMS['patches_per_image']),
    '--foreground_prob',   str(BEST_PARAMS['foreground_prob']),
    '--max_shapes',        str(BEST_PARAMS['max_shapes']),
    '--save_every',        '1',    # save epoch checkpoint every epoch
    '--keep_checkpoints',  '5',    # local: keep last 5 (saves Colab disk)
    '--drive_dir',         DRIVE_OUTPUT,  # Drive: keeps ALL epochs
]
if device == 'cuda':
    cmd.append('--amp')

# Stream output live AND capture for error reporting
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
captured = []
for line in proc.stdout:
    print(line, end='', flush=True)
    captured.append(line)
proc.wait()

if proc.returncode != 0:
    print(f'\nTraining failed (exit {proc.returncode})')
    print('--- last 50 lines ---')
    print(''.join(captured[-50:]))
else:
    log_path = os.path.join(LOCAL_OUTPUT, 'training_log.csv')
    if os.path.exists(log_path):
        with open(log_path) as f:
            rows = list(csv.DictReader(f))
        if rows:
            last = rows[-1]
            print(f'\nTraining complete!')
            print(f'  Final PSNR: {float(last.get("val_psnr", 0)):.2f} dB')
            print(f'  Final SSIM: {float(last.get("val_ssim", 0)):.4f}')
            print(f'  Final RMSE: {last.get("val_rmse", "n/a")}')
    print(f'\nCheckpoints on Drive: {DRIVE_OUTPUT}')
    print(f'  epoch_001.pth ... epoch_{EPOCHS:03d}.pth  (all epochs)')
    print(f'  best.pth                                   (best PSNR)')
    print(f'  training_log.csv                           (all metrics)')


In [ ]:
# 6. Plot training curves
log_path = os.path.join(LOCAL_OUTPUT, 'training_log.csv')
if os.path.exists(log_path):
    !python scripts/plot_training.py {log_path}
    from IPython.display import Image, display
    plot_files = glob.glob('checkpoints/training_curves*.png')
    if plot_files:
        display(Image(sorted(plot_files)[-1]))
else:
    print('No training log found yet.')